# SWI-Prolog

A self-contained refresher on **SWI-Prolog** — a mature, open-source Prolog system for logic programming and symbolic reasoning.

**Domain:** Symbolic AI & Logic  ·  **runnable:** yes (drives the `swipl` binary via `subprocess`)

## 1. What & Why

**Prolog** is a *declarative logic programming language*: you state **facts** and **rules**, then ask **queries**, and the engine searches for variable bindings that make the query provable. **SWI-Prolog** is the most widely used implementation — fast, ISO-compatible, batteries-included (constraint solvers, an HTTP server, tabling, a rich standard library), and free.

**The problem it solves.** When your problem is naturally expressed as *relations and inference rules* — "X is a grandparent of Z **if** X is a parent of some Y **and** Y is a parent of Z" — Prolog lets you write that almost verbatim. You describe *what* is true; the engine figures out *how* to derive answers via search and backtracking. No loops, no explicit data structures for the search frontier.

**Reach for it when:**
- You have rule-based / relational knowledge and want inference for free (expert systems, family/ontology reasoning, access-control policies).
- You're parsing structured text or building DSLs — Prolog's **DCG**s are a concise grammar formalism.
- You're solving combinatorial / constraint problems (scheduling, puzzles) with **CLP(FD)**.
- You want to prototype symbolic AI, theorem-proving-ish search, or graph reasoning quickly.

**Skip it when:** the work is numeric / ML-heavy, performance-critical in tight loops, or a straightforward data pipeline. Prolog's search model and dynamic typing make those awkward; use Python/NumPy or a dedicated solver instead.

## 2. Mental Model

> **Prolog = a small database of facts + rules, sitting under an automatic search engine.**

You never write control flow. You assert truths, and to answer a query the engine performs **depth-first search with backtracking** over the clause database, using **unification** (two-way pattern matching) to bind variables.

```
Query:  grandparent(tom, Who)?
           │
           ▼
   match rule head grandparent(X,Z) :- parent(X,Y), parent(Y,Z).
           │  unify X=tom
           ▼
   solve parent(tom, Y)  ──► Y = bob       (try first matching fact)
           │
           ▼
   solve parent(bob, Z)  ──► Z = ann
           │
           ▼
   success: Who = ann.   (on backtrack, retry other facts for more answers)
```

Two ideas do all the work:
- **Unification** — `X = ann` doesn't *assign*; it makes both sides equal by binding variables (`foo(X, b) = foo(a, Y)` ⟹ `X=a, Y=b`).
- **Backtracking** — when a branch fails (or you ask for more answers), the engine rewinds the most recent choice and tries the next clause.

## 3. Key Concepts

| Term | What it means |
|------|---------------|
| **Fact** | An unconditional truth: `parent(tom, bob).` |
| **Rule** | A conditional truth (Horn clause): `Head :- Body.` reads "Head is true *if* Body is true." |
| **Clause** | A fact or a rule. A **predicate** is the set of clauses with the same name/arity, e.g. `parent/2`. |
| **Atom vs Variable** | Lowercase = atom/constant (`tom`); Capitalized or `_` = logic variable (`X`, `_Who`). |
| **Unification** | Two-way pattern matching that binds variables to make terms equal. |
| **Backtracking / SLD resolution** | The depth-first proof search that drives query answering. |
| **Lists** | `[a, b, c]`, decomposed as `[Head | Tail]` — the bread and butter of recursion. |
| **`is/2`** | Arithmetic evaluation: `X is 2 + 3` binds `X=5`. Distinct from `=` (unify) and `=:=` (numeric equality). |
| **Cut `!`** | Commits to choices made so far, pruning backtracking. Powerful and easy to misuse. |
| **Negation as failure `\+`** | `\+ Goal` succeeds when `Goal` *cannot be proven* (closed-world assumption). |
| **`findall/3`, `bagof/3`, `setof/3`** | Collect all solutions of a goal into a list. |
| **`assert/retract`** | Add/remove clauses at runtime (mutating the database). |
| **DCG** | Definite Clause Grammars — `-->` notation for parsing/generation. |

## 4. Setup

SWI-Prolog is a native binary, so install it from your package manager:

- **macOS:** `brew install swi-prolog`
- **Debian/Ubuntu:** `sudo apt install swi-prolog`
- **Windows / others:** download from <https://www.swi-prolog.org/download/stable>

There's no `pip install` for the language itself. To *embed* Prolog in Python you can `pip install pyswip` (a `ctypes` binding), but it requires the native library present and adds setup friction. For a refresher, the most robust approach — and what we use below — is to call the `swipl` binary via `subprocess`: it needs nothing beyond the standard library.

The cell below probes for `swipl` and defines a tiny `run_prolog(program, goal)` helper. Everything is gated on `shutil.which("swipl")`, so the notebook still executes top-to-bottom even on a machine without SWI-Prolog installed (the example cells just print the program and a note).

In [1]:
import os
import shutil
import subprocess
import tempfile

SWIPL = shutil.which("swipl")
print("swipl found at:" , SWIPL or "(not installed)")


def run_prolog(program: str, goal: str, timeout: int = 30) -> str:
    """Consult `program`, run `goal`, return swipl's combined output.

    Gated on swipl being installed so the notebook runs either way.
    `goal` should print its own results (e.g. via format/2) and is run
    with `-t halt` so the process exits instead of dropping to a REPL.
    """
    if not SWIPL:
        return "[swipl not installed - showing program only]\n" + program
    with tempfile.NamedTemporaryFile("w", suffix=".pl", delete=False) as f:
        f.write(program)
        path = f.name
    try:
        proc = subprocess.run(
            [SWIPL, "-q", "-g", goal, "-t", "halt", path],
            capture_output=True, text=True, timeout=timeout,
        )
        return (proc.stdout + proc.stderr).strip()
    finally:
        os.unlink(path)


if SWIPL:
    print(subprocess.run([SWIPL, "--version"], capture_output=True, text=True).stdout.strip())


swipl found at: /opt/homebrew/bin/swipl
SWI-Prolog version 10.0.2 for arm64-darwin


## 5. Worked Examples

### Example 1 — Facts, rules, and backtracking

A classic family knowledge base. We assert `parent/2` facts, define `grandparent/2` and `sibling/2` as rules, and let the engine enumerate every answer by backtracking. Note how `sibling` excludes a person being their own sibling with `X \= Y`.

In [2]:
family = r"""
parent(tom,  bob).
parent(tom,  liz).
parent(bob,  ann).
parent(bob,  pat).
parent(pat,  jim).

grandparent(X, Z) :- parent(X, Y), parent(Y, Z).
sibling(X, Y)     :- parent(P, X), parent(P, Y), X \= Y.
"""

print("All grandparent pairs:")
print(run_prolog(family,
    "forall(grandparent(G, C), format('  ~w is grandparent of ~w~n', [G, C]))"))

print("\nSiblings of ann:")
print(run_prolog(family,
    "forall(sibling(ann, S), format('  ~w~n', [S]))"))


All grandparent pairs:
tom is grandparent of ann
  tom is grandparent of pat
  bob is grandparent of jim

Siblings of ann:
pat


### Example 2 — Recursion, lists, and collecting solutions

Prolog has no loops; you recurse over lists with `[H|T]`. Here `mylen/2` counts elements, `between/3` (built-in) generates a range, and `findall/3` gathers every solution of a goal into a list. We compute the squares of 1..5 as a one-liner over the relation `Y is X*X`.

In [3]:
lists = r"""
mylen([], 0).
mylen([_|T], N) :- mylen(T, N0), N is N0 + 1.

square(X, Y) :- Y is X * X.
"""

print("Length of [a,b,c,d]:")
print(run_prolog(lists, "mylen([a,b,c,d], N), format('  N = ~w~n', [N])"))

print("\nSquares of 1..5 collected with findall/3:")
print(run_prolog(lists,
    "findall(Y, (between(1,5,X), square(X,Y)), Sqs), format('  ~w~n', [Sqs])"))


Length of [a,b,c,d]:
N = 4

Squares of 1..5 collected with findall/3:
[1,4,9,16,25]


### Example 3 — A solver: N-Queens with CLP(FD)

SWI-Prolog ships `library(clpfd)` for **constraint logic programming over finite domains**. Instead of brute-force `generate-and-test`, you *post constraints* and let the propagation+`labeling` engine search efficiently. This places N non-attacking queens — declarative, and only a handful of lines.

In [4]:
queens = r""":- use_module(library(clpfd)).

n_queens(N, Qs) :-
    length(Qs, N),
    Qs ins 1..N,
    all_distinct(Qs),
    safe(Qs),
    labeling([ff], Qs).

safe([]).
safe([Q|Qs]) :- safe(Qs, Q, 1), safe(Qs).
safe([], _, _).
safe([Q|Qs], Q0, D0) :-
    Q0 #\= Q + D0,
    Q0 #\= Q - D0,
    D1 #= D0 + 1,
    safe(Qs, Q0, D1).
"""

print("One solution to the 8-queens problem (column per row):")
print(run_prolog(queens, "once(n_queens(8, Qs)), format('  ~w~n', [Qs])"))


One solution to the 8-queens problem (column per row):
[1,5,8,6,3,7,2,4]


## 6. Gotchas & Pitfalls

- **`=` is unification, not assignment.** `X = X + 1` does *not* increment — it tries to unify `X` with the term `X+1` and fails (or loops). Use `is/2` for arithmetic: `Y is X + 1`. And `is/2` needs the right side fully bound: `X is Y + 1` with `Y` unbound throws an instantiation error.
- **Three different "equals."** `=` (unify), `==` (already-identical terms, no binding), `=:=` (numeric value equality, evaluates both sides). Mixing them up is the #1 beginner bug.
- **Clause order matters.** Prolog tries clauses top-to-bottom. Put base cases first and recursive cases after, and beware **left recursion** (`path(X,Z) :- path(X,Y), edge(Y,Z).`) which loops forever — recurse on the *smaller* argument.
- **Singleton variable warnings.** A variable used once is usually a typo. Silence intentional ones with a leading underscore: `_` or `_Ignored`.
- **The cut `!` changes meaning, not just speed.** A "green" cut only prunes redundant work; a "red" cut also removes solutions. Adding a cut to fix performance can silently make your predicate incomplete.
- **Negation as failure is not logical negation.** `\+ Goal` means "Goal is *not provable* right now" under the **closed-world assumption**. `\+ p(X)` with `X` unbound is almost always a bug — it asks "is there *no* X at all," not "find an X that fails." Ground your variables before negating.
- **Floats vs integers.** `is/2` follows ISO rules; `5 / 2` is `2.5`, integer division is `5 // 2`. Use `=:=` not `=` to compare numbers.
- **No automatic occurs-check.** `X = f(X)` creates a cyclic term silently; use `unify_with_occurs_check/2` if that matters.

## 7. When to Use vs Alternatives

| Option | Sweet spot | Trade-off vs SWI-Prolog |
|--------|-----------|--------------------------|
| **SWI-Prolog** | General logic programming, parsing (DCG), CLP(FD), rule engines, prototyping symbolic search | Turing-complete with function symbols ⇒ termination is *your* responsibility; recursion can loop. |
| **[[datalog]] (Datalog)** | Recursive queries over relational data, deductive databases | Decidable & guaranteed to terminate (no function symbols), set semantics — but strictly less expressive. |
| **[[answer-set-programming]] (ASP / clingo)** | Combinatorial search, planning, problems with "choose a model" semantics | Stable-model semantics with true negation and disjunction; finds *all* models, not just first proof. Less suited to procedural/IO work. |
| **[[minikanren]] (miniKanren)** | Embedding relational logic *inside* a host language (Scheme/Python/Clojure) | Tiny and embeddable; complete (interleaving) search avoids Prolog's depth-first traps, but far smaller ecosystem. |
| **[[z3-smt]] (Z3 / SMT)** | Constraints over rich theories (ints, reals, bitvectors, arrays) | A decision procedure, not a programming language — great for satisfiability, not for general inference/recursion. |
| **[[clips]] / [[rete-algorithm]] (CLIPS, Rete)** | Forward-chaining production systems reacting to a growing fact base | Data-driven (facts trigger rules) vs Prolog's goal-driven backward chaining. |
| **[[problog]] (ProbLog)** | Logic + probabilities (uncertain facts/rules) | Adds probabilistic inference on top of a Prolog-like syntax; heavier and slower. |

**Rule of thumb:** if you want *backward-chaining goal resolution* with general recursion, real I/O, parsing, and constraint solving in one mature system, SWI-Prolog is the default. If you need *guaranteed termination* (Datalog), *all stable models* (ASP), or *theory-level constraint solving* (SMT), prefer the specialized tool.

## 8. Resources

- **Official site & downloads** — <https://www.swi-prolog.org/>
- **Reference manual** — <https://www.swi-prolog.org/pldoc/doc_for?object=manual>
- **The Power of Prolog** (Markus Triska) — the best modern, idiomatic tutorial, with deep CLP(FD) coverage: <https://www.metalevel.at/prolog>
- **Learn Prolog Now!** — gentle, classic introduction: <http://www.learnprolognow.org/>
- **CLP(FD) library docs** — <https://www.swi-prolog.org/pldoc/man?section=clpfd>